# 15 — Risk Management

Notebook này quản lý rủi ro danh mục (VaR, ES, Beta) và áp dụng các kịch bản dừng lỗ.

In [1]:
# Cell: Setup Project Root
import os
import sys
from pathlib import Path


def find_project_root(start_path=None):
    path = Path(start_path or os.getcwd()).resolve()
    for parent in [path, *path.parents]:
        if (parent / "src").is_dir() and (parent / "config").is_dir():
            return parent
    raise FileNotFoundError("Khong tim thay project root co thu muc src/ va config/")


project_root = find_project_root()
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)


Project root: /Users/jamesb/Desktop/vietnam-quant-research


In [2]:
# Cell: Import Risk Modules
import pandas as pd

from src.analytics.correlation import build_return_matrix
from src.data.loaders import read_parquet
from src.portfolio.construction import hierarchical_risk_parity_allocation, shrink_covariance
from src.reporting.chart_factory import drawdown_chart, equity_curve_chart, sharpe_stability_chart, weight_bar_chart
from src.risk.management import (
    apply_drawdown_overlay,
    apply_symbol_caps,
    apply_volatility_target,
    build_portfolio_risk_report,
    expanding_sharpe_series,
    portfolio_nav_series,
    portfolio_return_series,
    rolling_drawdown,
    rolling_sharpe_series,
)


In [ ]:
# Cell: Load Returns And Base Weights
import pandas as pd

from src.validation.freshness_check import load_upstream_artifact

signals = read_parquet("data/signals/setup_signals.parquet")
signals["date"] = pd.to_datetime(signals["date"])

# Reads notebooks/outputs/ (Đợt 3 NV1). Previously this was a bare
# os.path.exists() on data/signals/selected_portfolio_weights.parquet followed
# by a silent read -- once the Đợt 2 guard stopped notebook 14 from refreshing
# that path, this cell went on consuming a days-old copy with no warning.
# load_upstream_artifact() always prints the file's age, and anything older
# than max_age_days is called out explicitly instead of passing as current.
selected_weights_df, upstream_info = load_upstream_artifact(
    "notebooks/outputs/selected_portfolio_weights.parquet",
    producer="notebooks/04_watchlist_portfolio/14_portfolio_construction.ipynb",
    max_age_days=7.0,
)

if not selected_weights_df.empty:
    if "symbol" in selected_weights_df.columns:
        selected_weights_df = selected_weights_df.set_index("symbol")
    base_weights = selected_weights_df["weight"]
    candidate_symbols = base_weights.index.tolist()
    print(f"Base weights from Notebook 14 for {len(candidate_symbols)} symbols: {candidate_symbols}")
    if upstream_info["status"] == "stale":
        print("   -> KET QUA DUOI DAY DUA TREN TRONG SO CU. Chay lai notebook 14 truoc khi dung that.")
    signals = signals[signals["symbol"].isin(candidate_symbols)].copy()
else:
    print("Khong co trong so tu Notebook 14 -- fallback sang HRP tren toan universe (dung thiet ke goc).")
    base_weights = None

return_matrix = build_return_matrix(signals, return_col="return_1d")
return_matrix = return_matrix.replace([float("inf"), float("-inf")], pd.NA)
return_matrix = return_matrix.dropna(axis=1, how="all").copy()
valid_cols = [col for col in return_matrix.columns if return_matrix[col].dropna().nunique() > 1]
return_matrix = return_matrix[valid_cols].fillna(0.0)

if return_matrix.empty:
    raise ValueError("Khong con du lieu return hop le de build risk notebook")

cov_matrix = shrink_covariance(return_matrix.cov(), shrinkage=0.15)

if base_weights is None:
    base_weights = hierarchical_risk_parity_allocation(returns=return_matrix)
else:
    # Align base_weights to valid returns columns
    base_weights = base_weights.reindex(return_matrix.columns).fillna(0.0)
    # Re-normalize just in case
    total_w = base_weights.sum()
    if total_w > 0:
        base_weights = base_weights / total_w
    else:
        base_weights = pd.Series(1.0 / len(base_weights), index=base_weights.index)

display(base_weights.to_frame("base_weight"))
display(weight_bar_chart(base_weights, title="Base Portfolio Allocation"))


In [4]:
# Cell: Build Base Risk Report
base_returns = portfolio_return_series(return_matrix, base_weights)
base_nav = portfolio_nav_series(base_returns)
base_drawdown = rolling_drawdown(base_nav)
base_expanding_sharpe = expanding_sharpe_series(base_returns, min_periods=21)
base_rolling_sharpe = rolling_sharpe_series(base_returns, window=min(63, max(21, len(base_returns) // 3)))
base_report = build_portfolio_risk_report(return_matrix, base_weights)

display(pd.DataFrame([base_report]).T.rename(columns={0: "value"}))
sharpe_stability_chart(base_expanding_sharpe, base_rolling_sharpe, title="Base Portfolio Sharpe Stability")


,value
annual_return,0.265980
annual_volatility,0.136416
sharpe_ratio,1.797526
sortino_ratio,2.930394
max_drawdown,-0.143684
calmar_ratio,1.851147
var_95,-0.009801
cvar_95,-0.018259
expanding_sharpe_latest,1.797526
rolling_sharpe_latest,2.294265


In [ ]:
# Cell: Apply Risk Overlays
capped_weights = apply_symbol_caps(base_weights, max_weight=0.20)
vol_targeted_weights, vol_scale = apply_volatility_target(capped_weights, cov_matrix, target_volatility=0.12, max_gross_leverage=1.25)
current_drawdown = float(base_drawdown.iloc[-1])
overlay_weights, exposure_scale = apply_drawdown_overlay(vol_targeted_weights, current_drawdown=current_drawdown)

overlay_report = build_portfolio_risk_report(return_matrix, overlay_weights)
print("Vol target scale:", round(vol_scale, 4))
print("Drawdown overlay scale:", round(exposure_scale, 4))

# Writes to notebooks/outputs/, NOT data/signals/ (Đợt 3 NV1) -- same reasoning
# as notebook 14's save cell: pipeline-owned dir, zero automated readers.
from pathlib import Path

from src.data.loaders import save_parquet

NOTEBOOK_OUTPUTS = Path("notebooks/outputs")
NOTEBOOK_OUTPUTS.mkdir(parents=True, exist_ok=True)

overlay_df = overlay_weights.reset_index()
overlay_df.columns = ["symbol", "target_weight"]
save_parquet(overlay_df, NOTEBOOK_OUTPUTS / "risk_overlay_weights.parquet")
print("Saved risk overlay weights to notebooks/outputs/risk_overlay_weights.parquet")

pd.DataFrame([overlay_report]).T.rename(columns={0: "value"})


In [6]:
# Cell: Compare Equity And Drawdown
overlay_returns = portfolio_return_series(return_matrix, overlay_weights)
overlay_nav = portfolio_nav_series(overlay_returns)
overlay_drawdown = rolling_drawdown(overlay_nav)
overlay_expanding_sharpe = expanding_sharpe_series(overlay_returns, min_periods=21)
overlay_rolling_sharpe = rolling_sharpe_series(overlay_returns, window=min(63, max(21, len(overlay_returns) // 3)))

display(equity_curve_chart(base_nav, title="Base Portfolio NAV"))
display(equity_curve_chart(overlay_nav, title="Overlay Portfolio NAV"))
display(drawdown_chart(base_drawdown, title="Base Drawdown"))
display(drawdown_chart(overlay_drawdown, title="Overlay Drawdown"))
display(sharpe_stability_chart(base_expanding_sharpe, base_rolling_sharpe, title="Base Portfolio Sharpe Stability"))
sharpe_stability_chart(overlay_expanding_sharpe, overlay_rolling_sharpe, title="Overlay Portfolio Sharpe Stability")
